# TCC — comparativos das execuções definitivas

Somente leitura, sem treinamento. Configure EXECUCOES na primeira célula de código e clique em Executar tudo. Use TODAS ou os nomes completos das pastas separados por vírgula. Não há seleção manual obrigatória entre células. O Drive poderá pedir autorização.

## 1. Conectar e escolher a pasta
Preencha EXECUCOES antes de Executar tudo. Mantidas inicialmente as quatro execuções da última leitura. Para comparar todas, escreva TODAS. A tabela lista as pastas disponíveis e marca as selecionadas. Altere OUTPUT_ROOT apenas se moveu os resultados.

In [ ]:
#@title 1. Configurar execuções e conectar o Drive
# Use TODAS ou nomes completos de pastas separados por vírgula.
EXECUCOES = "ResNet50_20261028T093036_176792Z, ResNet50_20261027T215651_213059Z, EfficientNetB0_20261025T211725_513336Z, EfficientNetB0_20261027T092059_064925Z, InceptionResNetV2_20261022T205547_012854Z, InceptionResNetV2_20261026T095420_553995Z, DenseNet169_20261029T175413_205114Z, DenseNet169_20261029T000505_878389Z" #@param {type:"string"}

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from html import escape
from google.colab import drive
drive.mount('/content/drive')

# Altere somente se os resultados definitivos estiverem em outra pasta.
OUTPUT_ROOT = Path('/content/drive/MyDrive/TCC/resultados/tcc_consolidado_v1')
PASTA_EXECUCOES = OUTPUT_ROOT / 'execucoes'
NOMES_ANALISES = {'aleatorio':'Aleatório (teste 30%)',
    'blocos_central':'Blocos — teste central', 'blocos':'Blocos — teste final', 'loeo':'LOEO'}
PARTICOES = ('treino','validacao','teste')
COLUNAS_TEMPORAIS = (0,1,5,10,25,50,100)
EXPERIMENTOS = [f'L{i}' for i in range(1,10)]


# A listagem lê somente os manifestos. Nenhum gráfico aparece antes da seleção.
if not PASTA_EXECUCOES.is_dir():
    raise FileNotFoundError(f'Pasta não encontrada: {PASTA_EXECUCOES}. Confira o caminho e a montagem do Drive.')
opcoes = []
for arquivo in sorted(PASTA_EXECUCOES.glob('*/manifesto.json')):
    try:
        m = json.loads(arquivo.read_text()); c = m['config']
        label = (f"{c['arquitetura']} | parâmetros={c['parametros_processo']} | "
                 f"base={c['dataset_id']} | {arquivo.parent.name}")
        opcoes.append((label, arquivo.parent.name))
    except (OSError, ValueError, KeyError) as exc:
        print(f'Ignorado {arquivo.parent.name}: {exc}')


disponiveis = {nome for _, nome in opcoes}
nomes = [s.strip() for s in EXECUCOES.split(',') if s.strip()]
if EXECUCOES.strip().upper() == 'TODAS':
    nomes = sorted(disponiveis)
if not nomes:
    raise ValueError('Preencha EXECUCOES com TODAS ou com nomes de pastas.')
ausentes = set(nomes) - disponiveis
if ausentes:
    raise ValueError(f'Execuções não encontradas: {sorted(ausentes)}. Disponíveis: {sorted(disponiveis)}')
EXECUCOES_SELECIONADAS = set(nomes)
# Seleção já definida antes do Run All; não depende de interação com widgets.
seletor = widgets.SelectMultiple(options=opcoes, value=tuple(sorted(EXECUCOES_SELECIONADAS)))
display(pd.DataFrame([{'execucao':nome,'selecionada':nome in EXECUCOES_SELECIONADAS} for _,nome in opcoes]))
print(f'{len(EXECUCOES_SELECIONADAS)} execução(ões) selecionada(s). As próximas células podem rodar automaticamente.')


## 2. Gráficos para publicação no TCC
Execute esta célula de funções e depois carregue os resultados na seção 4. Cada gráfico continua em sua própria célula.

- Nomes curtos, cores fixas por CNN e distinção com/sem parâmetros.
- Todas as análises: painéis separados por protocolo, com a mesma escala vertical.
- Evolução temporal: opção de separar por CNN; média 42/43 e amplitude observada, não intervalo de confiança.
- Variação: MAE percentual em relação a K=0; R² como diferença absoluta.
- Botão de exportação nos painéis comparativos: PDF, SVG, PNG a 300 dpi, dados e filtros.
- Identificadores completos permanecem na rastreabilidade. Nenhum treinamento é executado.

In [ ]:

# Fontes ampliadas para todos os visuais e exportações.
plt.rcParams.update({'font.size':22,'axes.titlesize':26,'axes.labelsize':24,'xtick.labelsize':20,'ytick.labelsize':20,'legend.fontsize':20,'legend.title_fontsize':22,'figure.titlesize':28,'axes.titlepad':18,'axes.labelpad':12})
def medir(real,pred):
    real,pred=np.asarray(real).ravel(),np.asarray(pred).ravel()
    rmse=float(np.sqrt(np.mean((real-pred)**2))); desvio=float(np.std(real,ddof=0))
    return {'MAE_mm':float(np.mean(np.abs(real-pred))), 'MAPE_pct':float(100*np.mean(np.abs((real-pred)/real))),
            'RMSE_mm':rmse,'R2':float(1-np.mean((real-pred)**2)/desvio**2) if desvio>0 else np.nan,
            'desvio_y_mm':desvio,'RMSE_desvio':rmse/desvio if desvio>0 else np.nan,
            'vies_mm':float(np.mean(pred-real)),'n':len(real)}


# Painéis compartilhados — leem somente a nova pasta de execuções, sem treinar.
def carregar_resultados(raiz):
    linhas=[]; avisos=[]
    arquivos = [p for p in sorted(Path(raiz).glob('*/manifesto.json')) if p.parent.name in EXECUCOES_SELECIONADAS]
    for numero, arquivo in enumerate(arquivos, 1):
        print(f'Lendo {numero}/{len(arquivos)}: {arquivo.parent.name}', flush=True)
        if arquivo.parent.name not in EXECUCOES_SELECIONADAS: continue
        try:
            m=json.loads(arquivo.read_text()); cfg=m['config']; run=arquivo.parent
            experimentos = sorted(pd.read_csv(run/'indice_features.csv').experimento.unique())
            for etapa in ['estatico','temporal']:
                protos=cfg['protocolos'] if etapa=='estatico' else cfg['protocolos_temporais']
                ks=[0] if etapa=='estatico' else cfg['colunas']
                for proto in protos:
                    for seed in cfg['sementes']:
                        pasta=run/etapa/proto/f'seed_{seed}'
                        esperados=experimentos if proto=='loeo' else ['unico']
                        folds=[pasta/f for f in esperados]
                        if not all((f/'manifesto.json').exists() for f in folds):
                            avisos.append(f'{run.name} | {etapa}/{proto}/seed {seed}: pendente'); continue
                        for f in folds:
                            mf=json.loads((f/'manifesto.json').read_text())
                            if mf['assinatura']!=m['assinatura'] or ((f/'concluido.json').exists() and json.loads((f/'concluido.json').read_text())!=mf):
                                raise ValueError('Assinatura divergente: '+str(f))
                        for k in ks:
                            if not all((f/f'K_{k:03d}'/'concluido.json').exists() for f in folds):
                                avisos.append(f'{run.name} | {etapa}/{proto}/seed {seed}/K={k}: incompleto'); continue
                            pred=[]; metric=[]
                            for f in folds:
                                fp=f/f'K_{k:03d}'
                                if not (fp/'concluido.json').exists(): raise ValueError('K incompleto: '+str(fp))
                                d=pd.read_csv(fp/'predicoes.csv'); d['fold']=f.name; pred.append(d)
                                metric.append(pd.read_csv(fp/'metricas.csv'))
                            d=pd.concat(pred,ignore_index=True); metr=pd.concat(metric,ignore_index=True)
                            for conjunto,g in d.groupby('conjunto'):
                                if conjunto=='teste' and g.duplicated('indice_dataset').any(): raise ValueError('Teste duplicado.')
                                if not np.isfinite(g[['y','y_predito']]).all().all(): raise ValueError('Previsões inválidas.')
                                custos=metr.loc[metr.conjunto==conjunto]
                                linhas.append({'execucao':run.name,'pasta':str(run),'CNN':cfg['arquitetura'],
                                    'base':cfg['dataset_id'],'grupo':m['grupo'],'parametros':cfg['parametros_processo'],
                                    'etapa':etapa,'protocolo':proto,'semente':seed,'colunas':k,'conjunto':conjunto,
                                    **medir(g.y,g.y_predito),'custo_s':float((custos.tempo_fit_final_s+custos.tempo_auxiliar_acumulado_s).sum())})
        except (KeyError,ValueError,OSError) as exc: avisos.append(f'{arquivo}: {exc}')
    return pd.DataFrame(linhas),avisos

def grupo_visual_sem_parametros(pasta, etapa):
    # A flag de parâmetros é uma variável comparada, não um filtro de grupo.
    # Mantém base, configuração restante e índices efetivos das divisões.
    import hashlib
    run=Path(pasta)
    m=json.loads((run/'manifesto.json').read_text())
    cfg=m['config']
    ignorar={'arquitetura','origem_features','pooling','entrada','parametros_processo'}
    comum={k:v for k,v in cfg.items() if k not in ignorar}
    arquivo_divisao='estatico.npz' if etapa=='estatico' else 'pareado.npz'
    arquivos=sorted((run/'divisoes').glob(f'*/*/{arquivo_divisao}'))
    if not arquivos:
        raise ValueError(f'Divisões não encontradas em {run}; não é seguro juntar os grupos.')
    divisoes={}
    for arq in arquivos:
        with np.load(arq,allow_pickle=False) as z:
            divisoes[str(arq.relative_to(run/'divisoes'))]={s:z[s].tolist() for s in PARTICOES}
    dados={'base':m['base_hash'],'config':comum,'divisoes':divisoes}
    return hashlib.sha256(json.dumps(dados,sort_keys=True).encode()).hexdigest()[:16]


# Estilo de publicação: leitura apenas; exportação cria arquivos separados.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
CORES_TCC = {'EfficientNetB0':'#0072B2','InceptionResNetV2':'#D55E00',
             'ResNet50':'#009E73','DenseNet169':'#CC79A7'}
ORDEM_TCC = list(CORES_TCC)
NOMES_TCC = {'aleatorio':'Divisão aleatória','blocos_central':'Bloco central',
             'blocos':'Bloco final','loeo':'LOEO'}
METRICAS_TCC = {'MAE_mm':'MAE (mm)','MAPE_pct':'MAPE (%)','RMSE_mm':'RMSE (mm)',
                'R2':'R² global','RMSE_desvio':'RMSE / desvio dos rótulos'}

def nome_config_tcc(cnn, parametros):
    return str(cnn) + (' + parâmetros' if str(parametros).lower()=='true' else ' sem parâmetros')


def figura_evolucao_tcc(d, metrica, por_cnn=False, mostrar_amplitude=False):
    from matplotlib.ticker import MaxNLocator, AutoMinorLocator, ScalarFormatter
    d=d.copy().sort_values(['execucao','protocolo','semente','colunas'])
    keys=['execucao','protocolo','semente']
    if d.duplicated(keys+['colunas']).any(): raise ValueError('Resultados duplicados por K/semente.')
    ref=d[d.colunas==0][keys+[metrica]].rename(columns={metrica:'ref_zero'})
    d=d.merge(ref,on=keys,how='left',validate='many_to_one').sort_values(keys+['colunas'])
    if d.ref_zero.isna().any(): raise ValueError('K=0 ausente: comparação indisponível.')
    d['K_anterior']=d.groupby(keys).colunas.shift()
    d['ref_anterior']=d.groupby(keys)[metrica].shift()
    d['absoluto']=d[metrica]
    for nome,refcol in [('vs_zero','ref_zero'),('vs_anterior','ref_anterior')]:
        d[nome]=d[metrica]-d[refcol] if metrica=='R2' else 100*(d[metrica]-d[refcol])/d[refcol].replace(0,np.nan)
    cnns=[c for c in ORDEM_TCC if c in set(d.CNN)]+sorted(set(d.CNN)-set(ORDEM_TCC))
    protos=[p for p in NOMES_TCC if p in set(d.protocolo)]
    specs=[(p,c) for p in protos for c in cnns] if por_cnn else [(p,None) for p in protos]
    fig,axes=plt.subplots(len(specs),3,figsize=(21,7*len(specs)+2.4),squeeze=False)
    summaries=[]
    for row,(proto,cnn) in enumerate(specs):
        z=d[d.protocolo==proto]
        if cnn is not None: z=z[z.CNN==cnn]
        for col,(campo,titulo) in enumerate([('absoluto',METRICAS_TCC[metrica]),('vs_zero','Variação versus K=0'),('vs_anterior','Variação versus K anterior')]):
            ax=axes[row,col]
            g=z.groupby(['execucao','CNN','parametros','colunas'],as_index=False).agg(media=(campo,'mean'),minimo=(campo,'min'),maximo=(campo,'max'),n_sementes=(campo,'count'))
            g['protocolo']=proto; g['comparacao']=campo; summaries.append(g)
            for (run,c,param),a in g.groupby(['execucao','CNN','parametros']):
                a=a.sort_values('colunas'); color=CORES_TCC.get(c,'#555555')
                ax.plot(a.colunas,a.media,color=color,ls='-' if param else '--',marker='D' if param else 'o',ms=4,lw=1.6)
                if mostrar_amplitude: ax.vlines(a.colunas,a.minimo,a.maximo,color=color,alpha=.3)
            vals=(g[['minimo','maximo']].to_numpy().ravel() if mostrar_amplitude else g['media'].to_numpy()); vals=vals[np.isfinite(vals)]
            if len(vals):
                lo,hi=vals.min(),vals.max(); pad=max((hi-lo)*.10,max(abs(lo),abs(hi),1e-6)*.01)
                ax.set_ylim(lo-pad,hi+pad)
            if col: ax.axhline(0,color='#555555',lw=.9,ls='--')
            ax.set_title(NOMES_TCC[proto]+(' · '+cnn if cnn else '')+'\n'+titulo,fontsize=22)
            ax.set_ylabel(METRICAS_TCC[metrica] if col==0 else 'ΔR² (absoluto)' if metrica=='R2' else 'Variação (%)',fontsize=22)
            ax.set_xlabel('Colunas temporais K',fontsize=22)
            ax.set_xticks(sorted(d.colunas.unique())); ax.tick_params(axis='x',labelrotation=45,labelsize=20)
            ax.tick_params(axis='y',labelsize=20)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=8,min_n_ticks=5))
            ax.yaxis.set_minor_locator(AutoMinorLocator(2))
            formatter=ScalarFormatter(useOffset=False); formatter.set_scientific(False); ax.yaxis.set_major_formatter(formatter)
            ax.grid(which='major',axis='both',alpha=.35,lw=.7)
            ax.grid(which='minor',axis='y',alpha=.18,lw=.5,ls=':')
            ax.set_axisbelow(True); ax.spines[['top','right']].set_visible(False)
    handles=[Line2D([0],[0],color=CORES_TCC.get(c,'#555555'),lw=2,label=c) for c in cnns]
    handles += [Line2D([0],[0],color='#333333',ls='--',marker='o',label='Sem parâmetros'),Line2D([0],[0],color='#333333',ls='-',marker='D',label='Com parâmetros')]
    fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,.01),ncol=3,frameon=False,fontsize=16)
    fig.tight_layout(rect=(0,.12,1,1))
    return fig,pd.concat(summaries,ignore_index=True),d

def figura_tcc(d, etapa, tipo, metrica, por_cnn=False, mostrar_amplitude=False):
    if tipo in ['Evolução do MAE','Variação percentual']:
        return figura_evolucao_tcc(d,metrica,por_cnn,mostrar_amplitude)
    # d contém uma base/grupo e execuções explicitamente selecionadas.
    d=d.copy()
    if tipo=='Custo × MAE':
        d=d[np.isfinite(d.custo_s)&(d.custo_s>0)].copy()
    if d.empty: raise ValueError('Nenhum resultado finito com custo positivo.')
    chaves=['execucao','CNN','parametros','protocolo','colunas']
    if d.duplicated(chaves+['semente']).any():
        raise ValueError('Há métricas duplicadas por execução/protocolo/K/semente.')
    d['valor']=d[metrica]
    variacao=tipo=='Variação percentual'
    if variacao:
        chave=['execucao','protocolo','semente']
        ref=d[d.colunas==0][chave+[metrica]].rename(columns={metrica:'referencia'})
        d=d.merge(ref,on=chave,how='left',validate='many_to_one')
        if d.referencia.isna().any(): raise ValueError('K=0 ausente em uma das sementes.')
        d['valor']=d[metrica]-d.referencia if metrica=='R2' else np.where(
            d.referencia!=0,100*(d[metrica]/d.referencia-1),np.nan)
    r=d.groupby(chaves,as_index=False).agg(media=('valor','mean'),
        minimo=('valor','min'),maximo=('valor','max'),n_sementes=('semente','nunique'),
        custo_min=('custo_s',lambda v:v.mean()/60))
    protos=[p for p in NOMES_TCC if p in set(d.protocolo)]
    cnns=[c for c in ORDEM_TCC if c in set(d.CNN)]+sorted(set(d.CNN)-set(ORDEM_TCC))
    curvas=tipo in ['Evolução do MAE','Variação percentual']
    specs=[(p,c) for p in protos for c in cnns] if curvas and por_cnn else [(p,None) for p in protos]
    ncols=min(2,len(specs)); nrows=int(np.ceil(len(specs)/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(20,5.3*nrows+1.5),squeeze=False,sharey=True)
    configs=d[['execucao','CNN','parametros']].drop_duplicates()
    duplicadas=configs.duplicated(['CNN','parametros'],keep=False)
    codigos={run:f'E{i+1:02d}' for i,run in enumerate(sorted(configs.execucao))}
    nomes={a.execucao:nome_config_tcc(a.CNN,a.parametros)+(f' · {codigos[a.execucao]}' if dup else '')
           for a,dup in zip(configs.itertuples(),duplicadas)}
    ylabel=('ΔR² em relação a K = 0' if metrica=='R2' else 'Variação do MAE (%)' if metrica=='MAE_mm' else f'Variação de {METRICAS_TCC[metrica]} (%)') if variacao else METRICAS_TCC[metrica]
    usados=set()
    for i,(ax,(proto,cnn_painel)) in enumerate(zip(axes.flat,specs)):
        q=r[r.protocolo==proto]
        if cnn_painel is not None: q=q[q.CNN==cnn_painel]
        ax.set_title(f'({chr(97+i)}) {NOMES_TCC[proto]}' + (f' — {cnn_painel}' if cnn_painel else ''),loc='left',fontsize=22)
        ax.spines[['top','right']].set_visible(False)
        ax.grid(axis='y',alpha=.18); ax.set_axisbelow(True)
        if q.empty:
            ax.text(.5,.5,'Sem resultados',ha='center',transform=ax.transAxes); continue
        if curvas:
            ks=sorted(d.colunas.unique()); pos={k:j for j,k in enumerate(ks)}
            for (run,cnn,param),g in q.groupby(['execucao','CNN','parametros']):
                g=g.sort_values('colunas'); cor=CORES_TCC.get(cnn,'#555555'); usados.add(cnn)
                ax.plot([pos[k] for k in g.colunas],g.media,color=cor,
                        ls='-' if param else '--',marker='D' if param else 'o',ms=4,lw=1.7,label=nomes[run])
                # As marcas claras mostram resultados individuais das sementes.
                raw=d[(d.execucao==run)&(d.protocolo==proto)]
                for s,z in raw.groupby('semente'):
                    ax.scatter([pos[k] for k in z.colunas],z.valor,color=cor,alpha=.35,s=15,
                               marker='o' if int(s)==42 else 'x')
                ax.vlines([pos[k] for k in g.colunas],g.minimo,g.maximo,color=cor,alpha=.25,lw=1)
            ax.set_xticks(range(len(ks)),ks)
            ax.set_xlabel('Número de atributos temporais (K)')
        elif tipo=='Custo × MAE':
            for (run,cnn,param),g in q.groupby(['execucao','CNN','parametros']):
                cor=CORES_TCC.get(cnn,'#555555'); usados.add(cnn)
                ax.scatter(g.custo_min,g.media,color=cor,marker='D' if param else 'o',s=60,
                           edgecolors='white',linewidth=.5,label=nomes[run])
                ax.vlines(g.custo_min,g.minimo,g.maximo,color=cor,alpha=.5)
                if etapa=='temporal':
                    for a in g.itertuples(): ax.annotate(f'K={a.colunas}',(a.custo_min,a.media),xytext=(4,4),textcoords='offset points',fontsize=11,fontweight='normal')
            ax.set_xscale('log'); ax.set_xlabel('Treinamento acumulado (min; escala log)')
        else:
            # Duas barras por arquitetura; repetição de configuração usa código curto.
            ordem=configs.sort_values(['CNN','parametros','execucao'])
            xs=[]; labels=[]
            for j,cnn in enumerate(cnns):
                cfgs=ordem[ordem.CNN==cnn]; largura=.72/max(1,len(cfgs))
                for k,a in enumerate(cfgs.itertuples()):
                    g=q[q.execucao==a.execucao]
                    if g.empty: continue
                    if len(g)!=1: raise ValueError('Selecione um único K para as barras.')
                    z=g.iloc[0]; x=j-.36+(k+.5)*largura; cor=CORES_TCC.get(cnn,'#555555'); usados.add(cnn)
                    ax.bar(x,z.media,width=largura*.92,color=cor,alpha=1 if a.parametros else .45,
                           hatch=None if a.parametros else '//',edgecolor=cor,linewidth=.7)
                    ax.errorbar(x,z.media,yerr=[[z.media-z.minimo],[z.maximo-z.media]],fmt='none',ecolor='#333333',capsize=3,lw=.8)
                    ax.annotate(f'{z.media:.3f}',(x,z.maximo if z.media>=0 else z.minimo),xytext=(0,(5+13*(k%2)) if z.media>=0 else -(5+13*(k%2))),textcoords='offset points',ha='center',va='bottom' if z.media>=0 else 'top',fontsize=11,fontweight='normal',annotation_clip=False)
                xs.append(j); labels.append(cnn)
            ax.set_xticks(xs,labels,rotation=20,ha='right',fontsize=22)

        ax.set_ylabel(ylabel,fontsize=22)
        ax.tick_params(axis='y',labelsize=20)
        if variacao or metrica=='R2': ax.axhline(0,color='#555555',lw=.9)
        if metrica=='RMSE_desvio' and not variacao: ax.axhline(1,color='#555555',ls=':',lw=1)
        ax.margins(y=.14)
    limites=r[['minimo','maximo']].to_numpy().ravel()
    limites=limites[np.isfinite(limites)]
    if len(limites):
        lo,hi=float(limites.min()),float(limites.max())
        if not curvas and tipo!='Custo × MAE' and not variacao: lo=min(0,lo)
        if variacao or metrica=='R2': lo,hi=min(0,lo),max(0,hi)
        if metrica=='RMSE_desvio' and not variacao: hi=max(1,hi)
        margem=max((hi-lo)*.18,.05)
        axes.flat[0].set_ylim(lo-margem if lo<0 or variacao else max(0,lo-margem),hi+margem)
    for ax in list(axes.flat)[len(specs):]: ax.set_visible(False)
    handles=[Line2D([0],[0],color=CORES_TCC.get(c,'#555555'),lw=2,label=c) for c in cnns if c in usados]
    if curvas:
        handles += [Line2D([0],[0],color='#333333',ls='--',marker='o',label='Sem parâmetros'),Line2D([0],[0],color='#333333',ls='-',marker='D',label='Com parâmetros')]
    elif tipo=='Custo × MAE':
        handles += [Line2D([0],[0],color='#333333',ls='',marker='o',label='Sem parâmetros'),Line2D([0],[0],color='#333333',ls='',marker='D',label='Com parâmetros')]
    else:
        handles += [Patch(facecolor='white',edgecolor='#555555',hatch='//',label='Sem parâmetros'),Patch(facecolor='#555555',label='Com parâmetros')]
    fig.legend(handles=handles,loc='lower center',ncol=3,frameon=False,fontsize=22)
    fig.tight_layout(rect=(0,.12,1,.97))
    r['configuracao']=r.execucao.map(nomes)
    return fig,r,d


def tabelas_metricas_tcc(d, tabela, curvas=False):
    chaves=['execucao','CNN','parametros','protocolo','colunas']
    metricas=[c for c in ['MAE_mm','MAPE_pct','RMSE_mm','R2','desvio_y_mm','RMSE_desvio','vies_mm','n','custo_s'] if c in d.columns]
    resumo=d.groupby(chaves,as_index=False)[metricas].mean()
    sementes=d.groupby(chaves,as_index=False).agg(sementes=('semente',lambda x:', '.join(map(str,sorted(set(x))))))
    resumo=resumo.merge(sementes,on=chaves,validate='one_to_one')
    resumo['protocolo']=resumo.protocolo.map(NOMES_TCC)
    resumo['parametros']=resumo.parametros.map({True:'Com',False:'Sem'})
    nomes={'execucao':'Execução','CNN':'Arquitetura','parametros':'Parâmetros','protocolo':'Protocolo','colunas':'K','sementes':'Sementes','MAE_mm':'MAE (mm)','MAPE_pct':'MAPE (%)','RMSE_mm':'RMSE (mm)','R2':'R²','desvio_y_mm':'Desvio de y (mm)','RMSE_desvio':'RMSE/desvio','vies_mm':'Viés (mm)','n':'N (médio entre sementes)','custo_s':'Custo (s)'}
    vis=resumo.rename(columns=nomes)
    display(HTML('<h4>Valores das métricas — mesmas execuções e filtros do gráfico</h4><p>Quando selecionada a média 42/43, cada valor é a média das métricas das duas sementes, não uma métrica recalculada juntando previsões.</p><div style="font-size:20px;line-height:1.5;overflow-x:auto;max-height:480px;overflow-y:auto">'+vis.to_html(index=False,float_format=lambda v:f'{v:.4f}',escape=True)+'</div>'))
    if curvas and 'comparacao' in tabela.columns:
        for campo,titulo in [('vs_zero','Variação versus K=0'),('vs_anterior','Variação versus K anterior disponível')]:
            t=tabela[tabela.comparacao==campo].copy()
            t['parametros']=t.parametros.map({True:'Com',False:'Sem'})
            t['protocolo']=t.protocolo.map(NOMES_TCC)
            cols=chaves+['media','minimo','maximo','n_sementes']
            t=t[cols].rename(columns={**nomes,'media':'Variação média','minimo':'Mínimo','maximo':'Máximo','n_sementes':'Sementes válidas'})
            display(HTML('<h4>'+titulo+'</h4><p>Percentual para erros; diferença absoluta para R². NaN indica comparação indisponível.</p><div style="font-size:20px;line-height:1.5;overflow-x:auto;max-height:400px;overflow-y:auto">'+t.to_html(index=False,float_format=lambda v:f'{v:.4f}',escape=True)+'</div>'))
    return resumo

def painel_comparativo(etapa,tipo):
    if 'HISTORICO' not in globals() or HISTORICO.empty:
        print('Execute as seções 1 a 4 para carregar os resultados.'); return
    h=HISTORICO[(HISTORICO.etapa==etapa)&HISTORICO.execucao.isin(EXECUCOES_SELECIONADAS)].copy()
    if h.empty: print('Sem resultados desta etapa.'); return
    h['parametros']=h.parametros.map(lambda x:str(x).lower()=='true')
    grupos={p:grupo_visual_sem_parametros(p,etapa) for p in h.pasta.unique()}
    h['grupo_visual']=h.pasta.map(grupos)
    grupo=widgets.Dropdown(options=[(f'Base {g.base.iloc[0]} · grupo {key[:8]}',key) for key,g in h.groupby('grupo_visual')],description='Base/grupo:',layout=widgets.Layout(width='95%'))
    runs=widgets.SelectMultiple(description='Execuções:',rows=8,layout=widgets.Layout(width='95%'))
    proto=widgets.Dropdown(options=[('Todas as análises','todas')]+[(NOMES_TCC[p],p) for p in NOMES_TCC if p in set(h.protocolo)],description='Protocolo:')
    seed=widgets.Dropdown(options=[('Média 42/43 + amplitude','media'),('Semente 42','42'),('Semente 43','43')],description='Sementes:',layout=widgets.Layout(width='300px'))
    split=widgets.Dropdown(options=['teste','validacao','treino'],description='Conjunto:')
    met=widgets.Dropdown(options=[(v,k) for k,v in METRICAS_TCC.items()],value='RMSE_desvio' if tipo=='RMSE/desvio' else 'MAE_mm',description='Métrica:')
    kval=widgets.Dropdown(options=list(COLUNAS_TEMPORAIS) if etapa=='temporal' else [0],value=0,description='K:')
    porcnn=widgets.Checkbox(value=False,description='Separar curvas por CNN')
    exportar=widgets.Button(description='Exportar PDF, SVG e PNG',icon='download')
    out=widgets.Output(); status=widgets.Output(); estado={}
    curvas=tipo in ['Evolução do MAE','Variação percentual']
    kval.disabled=curvas
    porcnn.disabled=not curvas
    amplitude=widgets.Checkbox(value=False,description='Mostrar amplitude entre sementes',layout=widgets.Layout(width='350px'))
    amplitude.disabled=not curvas
    def preencher(_=None):
        cfg=h[h.grupo_visual==grupo.value][['execucao','CNN','parametros']].drop_duplicates().sort_values(['CNN','parametros','execucao'])
        dup=cfg.duplicated(['CNN','parametros'],keep=False)
        runs.options=[(nome_config_tcc(a.CNN,a.parametros)+(f' · E{i+1:02d}' if repetida else ''),a.execucao) for i,(a,repetida) in enumerate(zip(cfg.itertuples(),dup))]
        runs.value=tuple(cfg.execucao)
    def desenhar(_=None):
        with out:
            clear_output(wait=True)
            anterior=estado.pop('fig',None)
            if anterior is not None: plt.close(anterior)
            estado.clear()
            d=h[(h.grupo_visual==grupo.value)&h.execucao.isin(runs.value)&(h.conjunto==split.value)].copy()
            if proto.value!='todas': d=d[d.protocolo==proto.value]
            if seed.value=='media':
                d=d.groupby(['execucao','protocolo','colunas'],group_keys=False).filter(lambda g:set(g.semente)=={42,43})
            else: d=d[d.semente==int(seed.value)]
            if not curvas: d=d[d.colunas==kval.value]
            if d.empty: print('Sem resultados completos para estes filtros.'); return
            try:
                with plt.rc_context({'font.size':22,'axes.titlesize':26,'axes.labelsize':24,'xtick.labelsize':20,'ytick.labelsize':20,'legend.fontsize':20,'svg.fonttype':'none','pdf.fonttype':42}):
                    fig,tabela,brutos=figura_tcc(d,etapa,tipo,met.value,porcnn.value,amplitude.value)
                display(fig); plt.close(fig)
                resumo_metricas=tabelas_metricas_tcc(d,tabela,curvas)
                nota=f'{etapa} | {split.value} | '+('Média das métricas das sementes 42 e 43; amplitude mínimo–máximo, não intervalo de confiança.' if seed.value=='media' else f'Semente {seed.value}.')
                if curvas: nota+=' K em escala numérica real. Três painéis: métrica original, variação versus K=0 e versus K anterior disponível da mesma execução/semente. K=0 não tem anterior; referência zero gera variação percentual indefinida. Variações calculadas por semente antes da média. Eixos Y ajustados aos valores de cada painel.'
                if tipo=='Variação percentual': nota+= ' ΔR² positivo = melhora.' if met.value=='R2' else 'Referência K=0 pareada: abaixo de zero = redução do erro; acima = aumento.'
                if tipo=='Custo × MAE': nota+=' Custo: auxiliares até K + regressor final; sem extração CNN. Tempos não são benchmark controlado.'
                print(nota)
                print('Protocolos têm conjuntos de avaliação distintos. Escolha de K orientada pela validação. Confira os limites Y antes de comparar painéis.')
                display(HTML('<details><summary>Rastreabilidade e valores exatos</summary>'+tabela.to_html(index=False,float_format=lambda v:f'{v:.4f}',escape=True)+'</details>'))
                estado.update(fig=fig,tabela=tabela,resumo_metricas=resumo_metricas,brutos=brutos,nota=nota,filtros={'etapa':etapa,'tipo':tipo,'metrica':met.value,'grupo':grupo.value,'protocolo':proto.value,'semente':seed.value,'conjunto':split.value,'K':int(kval.value),'execucoes':list(runs.value)})
            except (ValueError,KeyError,OSError) as exc:
                print('Não foi possível desenhar:',exc)
    def salvar(_):
        from datetime import datetime,timezone
        with status:
            clear_output(wait=True)
            if not estado: print('Gere um gráfico válido antes de exportar.'); return
            pasta=OUTPUT_ROOT/'comparacoes'/'figuras_tcc'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
            pasta.mkdir(parents=True,exist_ok=False)
            for ext in ['pdf','svg','png']: estado['fig'].savefig(pasta/f'figura.{ext}',dpi=300,bbox_inches='tight',facecolor='white')
            estado['tabela'].to_csv(pasta/'resumo.csv',index=False)
            estado['resumo_metricas'].to_csv(pasta/'todas_metricas.csv',index=False)
            estado['brutos'].to_csv(pasta/'metricas_por_semente.csv',index=False)
            (pasta/'filtros.json').write_text(json.dumps(estado['filtros'],ensure_ascii=False,indent=2),encoding='utf-8')
            (pasta/'legenda.txt').write_text(estado['nota'],encoding='utf-8')
            print('Exportado em:',pasta)
    preencher(); grupo.observe(preencher,names='value')
    for w in [grupo,runs,proto,seed,split,met,kval,porcnn,amplitude]: w.observe(desenhar,names='value')
    exportar.on_click(salvar)
    display(widgets.VBox([grupo,runs,widgets.HBox([proto,seed]),widgets.HBox([split,met,kval]),widgets.HBox([porcnn,amplitude]),exportar,status,out]))
    desenhar()

def painel_curvas():
    if not EXECUCOES_SELECIONADAS:
        raise RuntimeError('Carregue as execuções na célula 4 primeiro.')
    fontes=[]
    for arq in sorted((OUTPUT_ROOT/'execucoes').glob('*/manifesto.json')):
        if arq.parent.name not in EXECUCOES_SELECIONADAS: continue
        m=json.loads(arq.read_text()); cfg=m['config']
        for p in cfg['protocolos_temporais']:
            for seed in cfg['sementes']:
                local=arq.parent/'temporal'/p/f'seed_{seed}'/'unico'
                if (local/'features_temporais.npy').exists(): fontes.append((nome_config_tcc(cfg['arquitetura'],cfg['parametros_processo'])+f' · {NOMES_TCC.get(p,p)} · semente {seed}',str(local)))
    if not fontes: print('As curvas estarão disponíveis após a geração temporal.'); return
    fonte=widgets.Dropdown(options=fontes,description='Execução:',layout=widgets.Layout(width='98%'))
    exp=widgets.Dropdown(options=EXPERIMENTOS,description='Experimento:')
    coluna=widgets.IntSlider(value=1,min=1,max=max(COLUNAS_TEMPORAIS),description='Coluna:',continuous_update=False)
    out=widgets.Output()
    def desenhar(_=None):
        with out:
            clear_output(wait=True); local=Path(fonte.value); run=local.parents[3]; proto=local.parents[1].name
            base=pd.read_csv(run/'indice_features.csv'); t=np.load(local/'features_temporais.npy',mmap_mode='r',allow_pickle=False)
            if t.ndim != 2 or t.shape[0] != len(base):
                raise ValueError('Features temporais incompatíveis com o índice salvo.')
            if coluna.value > t.shape[1]:
                print(f'Esta execução contém apenas {t.shape[1]} colunas. Reduza o seletor.'); return
            with np.load(run/'divisoes'/proto/'unico'/'pareado.npz',allow_pickle=False) as z: div={s:z[s] for s in PARTICOES}
            fig,axes=plt.subplots(3,1,figsize=(13,8),sharex=True)
            for ax,s in zip(axes,PARTICOES):
                ids=div[s]; ids=ids[base.iloc[ids].experimento.to_numpy()==exp.value]; ids=ids[np.argsort(base.iloc[ids].frame)]
                ax.scatter(base.iloc[ids].frame,base.iloc[ids].y,s=5,label='y do frame atual',color='#17324d')
                ax.scatter(base.iloc[ids].frame,t[ids,coluna.value-1],s=4,alpha=.6,label=f'Feature temporal {coluna.value}',color='#cc8b38')
                ax.set_title(s,loc='left'); ax.set_ylabel('Largura (mm)'); ax.grid(alpha=.2); ax.legend(fontsize=22)
            axes[-1].set_xlabel('Número do frame'); fig.suptitle(f'{NOMES_ANALISES[proto]} | {exp.value} | geração iterativa')
            fig.tight_layout(); plt.show(); plt.close(fig)
            print('A coluna j é previsão de t−1 de um auxiliar com j−1 colunas; não é simplesmente a medição de t−j.')
    for w in [fonte,exp,coluna]: w.observe(desenhar,names='value')
    display(widgets.VBox([fonte,widgets.HBox([exp,coluna]),out])); desenhar()


## 3. Conferir execuções configuradas
Esta célula apenas confirma a seleção da primeira célula; não interrompe Executar tudo.

In [ ]:
# Conferência da seleção feita na primeira célula. Não solicita interação.
print('Execuções configuradas:')
for nome in sorted(EXECUCOES_SELECIONADAS):
    print(' •', nome)
print('Para mudar, edite EXECUCOES na primeira célula e execute tudo novamente.')


## 4. Carregar resultados selecionados
Execute depois da seleção. A leitura mostra o progresso por execução e ocorre uma única vez. Ks concluídos podem ser vistos mesmo que os Ks seguintes ainda estejam pendentes. Resultados incompletos de um K não são comparados.

In [ ]:
# 4 — CARREGAR RESULTADOS. Sem treinamento e sem gráficos nesta célula.
if not seletor.value:
    raise ValueError('Selecione pelo menos uma execução na célula anterior.')
EXECUCOES_SELECIONADAS = set(seletor.value)
HISTORICO, AVISOS = carregar_resultados(PASTA_EXECUCOES)
print(f'Carregamento concluído: {len(HISTORICO)} registros de métricas.', flush=True)
if AVISOS:
    display(HTML('<details><summary>Pendências e arquivos ignorados</summary><pre>'+escape('\n'.join(AVISOS))+'</pre></details>'))
if HISTORICO.empty:
    print('Nenhum K completo encontrado nas execuções selecionadas. Consulte os avisos acima.')
else:
    display(HISTORICO.groupby(['CNN','etapa','protocolo','semente']).agg(Ks=('colunas',lambda x: sorted(set(x)))).reset_index())
    print('Agora execute apenas as células dos gráficos desejados. Elas reutilizam estes dados em memória.')


## 5. Estático — barras de métricas
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 5. Estático — barras de métricas
painel_comparativo('estatico', 'Barras')

## 6. Estático — custo × erro
Cores identificam a CNN; símbolos distinguem com e sem parâmetros. Selecione a métrica. Custo em minutos, sem extração CNN.

In [ ]:
# 6. Estático — custo × MAE (cor: R²)
painel_comparativo('estatico', 'Custo × MAE')

## 7. Estático — RMSE/desvio dos rótulos
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 7. Estático — RMSE/desvio dos rótulos
painel_comparativo('estatico', 'RMSE/desvio')

## 8. Temporal — evolução do MAE com K
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 8. Temporal — evolução do MAE com K
painel_comparativo('temporal', 'Evolução do MAE')

## 9. Temporal — variação percentual do MAE
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 9. Temporal — variação percentual do MAE
painel_comparativo('temporal', 'Variação percentual')

## 10. Temporal — barras de métricas
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 10. Temporal — barras de métricas
painel_comparativo('temporal', 'Barras')

## 11. Temporal — custo × erro
Selecione K e a métrica. Cores identificam a CNN; símbolos distinguem com e sem parâmetros. Custo: auxiliares até K + regressor final, sem extração CNN.

In [ ]:
# 11. Temporal — custo × MAE (cor: R²)
painel_comparativo('temporal', 'Custo × MAE')

## 12. Temporal — RMSE/desvio dos rótulos
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 12. Temporal — RMSE/desvio dos rótulos
painel_comparativo('temporal', 'RMSE/desvio')

## 13. Temporal — rótulo × feature por experimento
Execute esta célula para abrir somente este gráfico. Use os seletores exibidos abaixo para ajustar a comparação.

In [ ]:
# 13. Temporal — rótulo × feature por experimento
painel_curvas()

In [ ]:
#@title 13b. Estático — largura real × prevista por experimento
# Execute a configuração inicial. Apenas lê previsões salvas; não treina.
def painel_curvas_estatico():
    fontes=[]; avisos=[]
    for nome in sorted(EXECUCOES_SELECIONADAS):
        run=PASTA_EXECUCOES/nome
        try:
            manifesto=json.loads((run/'manifesto.json').read_text()); cfg=manifesto['config']
            for arq in sorted((run/'estatico').glob('*/*/*/K_000/predicoes.csv')):
                fold=arq.parent.parent; seed=fold.parent.name; proto=fold.parent.parent.name
                if not (arq.parent/'concluido.json').is_file(): continue
                mf=json.loads((fold/'manifesto.json').read_text())
                if mf['assinatura']!=manifesto['assinatura']:
                    avisos.append(f'Assinatura divergente: {arq}'); continue
                label=nome_config_tcc(cfg['arquitetura'],cfg['parametros_processo'])+f" · {NOMES_TCC.get(proto,proto)} · {seed} · fold {fold.name}"
                fontes.append((label,str(arq)))
        except (OSError,ValueError,KeyError) as exc:
            avisos.append(f'{nome}: {exc}')
    if avisos: print('\n'.join(avisos))
    if not fontes:
        print('Nenhuma previsão estática concluída nas execuções selecionadas.'); return
    fonte=widgets.Dropdown(options=fontes,description='Execução:',layout=widgets.Layout(width='98%'))
    exp=widgets.Dropdown(options=EXPERIMENTOS,description='Experimento:')
    saida=widgets.Output(); cache={}
    def desenhar(_=None):
        with saida:
            clear_output(wait=True)
            try:
                arquivo=Path(fonte.value)
                if fonte.value not in cache:
                    d=pd.read_csv(arquivo)
                    if not {'experimento','frame'}.issubset(d.columns):
                        run=arquivo.parents[5]
                        indice=pd.read_csv(run/'indice_features.csv')
                        faltantes=[c for c in ['experimento','frame'] if c not in d.columns]
                        d=d.merge(indice[['indice_dataset']+faltantes],on='indice_dataset',how='left',validate='many_to_one')
                    obrigatorias={'conjunto','experimento','frame','y','y_predito'}
                    if not obrigatorias.issubset(d.columns):
                        raise ValueError(f'Colunas ausentes: {obrigatorias-set(d.columns)}')
                    cache.clear(); cache[fonte.value]=d
                d=cache[fonte.value]
                fig,axes=plt.subplots(3,1,figsize=(13,8),sharex=True,sharey=True)
                for ax,conjunto in zip(axes,PARTICOES):
                    g=d.loc[(d.experimento==exp.value)&(d.conjunto==conjunto)].sort_values('frame')
                    ax.set_title(conjunto,loc='left'); ax.set_ylabel('Largura (mm)'); ax.grid(alpha=.2)
                    if g.empty:
                        ax.text(.5,.5,'Experimento ausente desta partição neste fold',ha='center',transform=ax.transAxes); continue
                    if g.duplicated('frame').any(): raise ValueError('Frames duplicados na partição; confira o arquivo.')
                    ax.scatter(g.frame,g.y,s=6,color='#17324d',label='Largura real')
                    ax.scatter(g.frame,g.y_predito,s=5,alpha=.65,color='#cc8b38',label='Previsão estática')
                    ax.legend(fontsize=22)
                axes[-1].set_xlabel('Número do frame — ordenado apenas para visualização')
                fig.suptitle(f'Estático | {exp.value} | '+fonte.label,fontsize=22,wrap=True)
                fig.tight_layout(); plt.show(); plt.close(fig)
                print('LOEO: selecione o fold correspondente ao experimento de teste desejado. Não juntamos previsões de treino de folds distintos.')
            except (OSError,ValueError,KeyError) as exc:
                plt.close('all'); print(f'Não foi possível desenhar: {exc}')
    fonte.observe(desenhar,names='value'); exp.observe(desenhar,names='value')
    display(widgets.VBox([fonte,exp,saida])); desenhar()

painel_curvas_estatico()


In [ ]:
#@title 14. Distribuição e dispersão dos rótulos por experimento
# Execute a célula 1 antes desta célula para conectar o Drive e selecionar execuções.
# Cada execução é analisada separadamente; não misture rótulos de bases diferentes.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

if 'PASTA_EXECUCOES' not in globals() or 'EXECUCOES_SELECIONADAS' not in globals():
    raise RuntimeError('Execute primeiro a célula 1: configurar execuções e conectar o Drive.')

opcoes_largura = sorted(
    nome for nome in EXECUCOES_SELECIONADAS
    if (PASTA_EXECUCOES / nome / 'indice_features.csv').is_file()
)
if not opcoes_largura:
    raise FileNotFoundError('Nenhuma execução selecionada contém indice_features.csv.')

escolha_largura = widgets.Dropdown(
    options=opcoes_largura, description='Execução:',
    layout=widgets.Layout(width='95%'),
    style={'description_width': 'initial'}
)
saida_largura = widgets.Output()

def mostrar_dispersao_larguras(alteracao=None):
    with saida_largura:
        clear_output(wait=True)
        nome = escolha_largura.value
        caminho = PASTA_EXECUCOES / nome / 'indice_features.csv'
        dados = pd.read_csv(caminho)
        obrigatorias = {'experimento', 'y'}
        if not obrigatorias.issubset(dados.columns):
            print('Colunas necessárias ausentes:', sorted(obrigatorias - set(dados.columns)))
            return
        if {'experimento', 'frame'}.issubset(dados.columns) and dados.duplicated(['experimento', 'frame']).any():
            raise ValueError('Há frames duplicados no índice da execução. Confira a base antes de plotar.')
        dados = dados[['experimento', 'y']].copy()
        dados['y'] = pd.to_numeric(dados['y'], errors='coerce')
        dados = dados.dropna(subset=['experimento', 'y'])
        dados = dados[np.isfinite(dados['y'])]
        ordem = [f'L{i}' for i in range(1, 10) if (dados['experimento'] == f'L{i}').any()]
        if not ordem:
            print('Nenhum rótulo válido encontrado para L1 a L9.')
            return
        grupos = [dados.loc[dados['experimento'] == exp, 'y'].to_numpy() for exp in ordem]
        fig, ax = plt.subplots(figsize=(11, 5))
        caixas = ax.boxplot(grupos, tick_labels=ordem, patch_artist=True, showfliers=False)
        for caixa in caixas['boxes']:
            caixa.set(facecolor='#b7d8e8', edgecolor='#416b82')
        for mediana in caixas['medians']:
            mediana.set(color='#b54b35', linewidth=2)
        ax.set(xlabel='Experimento', ylabel='Largura do cordão (mm)',
               title='Distribuição das larguras por experimento')
        ax.grid(axis='y', linestyle='--', alpha=0.35)
        fig.tight_layout()
        plt.show()
        resumo = dados.groupby('experimento')['y'].agg(
            amostras='count', media_mm='mean', mediana_mm='median',
            desvio_padrao_mm=lambda x: x.std(ddof=0),
            variancia_mm2=lambda x: x.var(ddof=0),
            minimo_mm='min', maximo_mm='max'
        ).reindex(ordem)
        print('Execução:', nome)
        print('Base completa da execução; pontos extremos ocultados apenas no gráfico.')
        print('Desvio-padrão e variância populacionais (ddof=0).')
        display(resumo.round(4))

escolha_largura.observe(mostrar_dispersao_larguras, names='value')
display(escolha_largura, saida_largura)
mostrar_dispersao_larguras()


In [ ]:
#@title 15. Painéis adicionais — custos, ganho temporal e sementes (somente leitura)
# Requer a célula de funções compartilhadas. Não treina nem modifica resultados.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display, clear_output

def painel_adicional_tcc():
    if 'HISTORICO' not in globals() or 'EXECUCOES_SELECIONADAS' not in globals():
        print('Execute as seções 1 a 4 para configurar e carregar os resultados.'); return
    h = HISTORICO.loc[HISTORICO.execucao.isin(EXECUCOES_SELECIONADAS)].copy()
    if h.empty:
        print('Nenhuma execução completa disponível.'); return
    h['grupo']=[grupo_visual_sem_parametros(p,e) for p,e in zip(h.pasta,h.etapa)]
    nomes_curto={a.execucao:nome_config_tcc(a.CNN,a.parametros) for a in h[['execucao','CNN','parametros']].drop_duplicates().itertuples()}
    grupo = widgets.Dropdown(options=[(f'Base {g.base.iloc[0]} · grupo {key[:8]}', key) for key,g in h.groupby('grupo')], description='Base/grupo:', layout=widgets.Layout(width='98%'))
    runs = widgets.SelectMultiple(description='Execuções:', rows=5, layout=widgets.Layout(width='98%'))
    proto = widgets.Dropdown(options=sorted(h.protocolo.unique()), description='Protocolo:')
    seed = widgets.Dropdown(options=[42,43], description='Semente:')
    kval = widgets.Dropdown(options=[0,1,5,10,25,50,100], value=10, description='K:')
    modo = widgets.Dropdown(options=['Extração por CNN','Custo temporal por etapa','Custo acumulado por K','Ganho temporal por experimento','Comparação entre sementes'], description='Visual:', layout=widgets.Layout(width='450px'))
    out = widgets.Output()
    def preencher(_=None):
        d=h[h.grupo==grupo.value][['execucao','CNN']].drop_duplicates()
        runs.options=[(nomes_curto[r.execucao],r.execucao) for r in d.itertuples()]
        runs.value=tuple(d.execucao)
    def desenhar(_=None):
        # Estilo local: não herda as fontes grandes dos outros painéis.
        with out, plt.rc_context({'font.size':11, 'font.weight':'normal',
                'axes.titlesize':13, 'axes.titleweight':'normal', 'axes.labelsize':12,
                'axes.labelweight':'normal', 'xtick.labelsize':10, 'ytick.labelsize':10,
                'legend.fontsize':10, 'legend.title_fontsize':11,
                'axes.titlepad':12, 'axes.labelpad':8}):
            clear_output(wait=True)
            d=h[(h.grupo==grupo.value)&h.execucao.isin(runs.value)&(h.protocolo==proto.value)]
            if d.empty: print('Sem execuções completas para os filtros.'); return
            print('Grupo compatível; hardware de treinamento pode diferir entre sessões. Tempos registrados não são benchmark controlado.')
            try:
                if modo.value=='Extração por CNN':
                    rows=[]
                    for pasta in d.pasta.unique():
                        a=pd.read_csv(Path(pasta)/'custo_extrator.csv').iloc[0].to_dict()
                        a['origem']=a.get('origem_features',pasta); rows.append(a)
                    r=pd.DataFrame(rows).drop_duplicates('origem')
                    labels=[f"{a.arquitetura} · extração {i+1}" for i,a in enumerate(r.itertuples())]
                    fig,axs=plt.subplots(1,2,figsize=(15,max(4.5,len(r)*.55)),layout="constrained")
                    for ax,col,fator,titulo in zip(axs,['tempo_extracao_s','ms_imagem_lote'],[1/60,1],['Extração (min)','Tempo amortizado (ms/imagem)']):
                        ax.barh(labels,r[col]*fator,color='#44779c'); ax.set_xlabel(titulo); ax.grid(axis='x',alpha=.2)
                    plt.show(); plt.close(fig)
                    display(r[['arquitetura','dispositivo','batch_extracao','entrada','dtype','memoria_features_MiB','origem']])
                    print('Inclui leitura/pré-processamento/CNN. Extrações reutilizadas contadas uma vez por origem. Não mede latência em tempo real.'); return
                t=d[(d.etapa=='temporal')&(d.semente==seed.value)]
                if modo.value in ['Custo temporal por etapa','Custo acumulado por K']:
                    rows=[]
                    for a in t[t.conjunto=='teste'].itertuples():
                        path=Path(a.pasta)/'temporal'/a.protocolo/f'seed_{a.semente}'/'unico'/f'K_{a.colunas:03d}'/'metricas.csv'
                        m=pd.read_csv(path); m=m[m.conjunto=='teste'].iloc[0]
                        rows.append(dict(execucao=a.execucao,CNN=a.CNN,K=a.colunas,aux=m.tempo_auxiliar_acumulado_s/60,final=m.tempo_fit_final_s/60))
                    r=pd.DataFrame(rows)
                    if r.empty: print('Sem custos temporais completos.'); return
                    fig,ax=plt.subplots(figsize=(14,6),layout="constrained")
                    if modo.value=='Custo temporal por etapa':
                        r=r[r.K==kval.value]
                        if r.empty: plt.close(fig); print('K indisponível.'); return
                        labels=[nomes_curto[a.execucao] for a in r.itertuples()]
                        ax.barh(labels,r.aux,label='Construção pelos auxiliares',color='#44779c')
                        ax.barh(labels,r.final,left=r.aux,label='Regressor final',color='#d49b56')
                        ax.set_xlabel('Tempo (min)'); ax.set_title(f'K={kval.value} | semente {seed.value}')
                    else:
                        for run,g in r.groupby('execucao'):
                            g=g.sort_values('K'); ax.plot(g.K,g.aux,'o-',label=nomes_curto[run],color=CORES_TCC.get(g.CNN.iloc[0],'#555555'))
                        ax.set_xlabel('K'); ax.set_ylabel('Construção acumulada dos auxiliares (min)')
                    ax.legend(fontsize=10,loc='upper left',bbox_to_anchor=(1.02,1),borderaxespad=0,frameon=False); ax.grid(alpha=.2); plt.show(); plt.close(fig); display(r)
                    print('Sem extração CNN. As etapas até K são reutilizadas; não somar as curvas para obter o custo total da busca.'); return
                if modo.value=='Ganho temporal por experimento':
                    rows=[]
                    for run,g in t.groupby('execucao'):
                        v=g[g.conjunto=='validacao'].sort_values(['MAE_mm','colunas'])
                        if set(v.colunas) != set(COLUNAS_TEMPORAIS):
                            print('Grade K incompleta; ganho não calculado:', run); continue
                        k=int(v.iloc[0].colunas)
                        root=Path(g.pasta.iloc[0])/'temporal'/proto.value/f'seed_{seed.value}'/'unico'
                        p0=pd.read_csv(root/'K_000'/'predicoes.csv'); pk=pd.read_csv(root/f'K_{k:03d}'/'predicoes.csv')
                        p0=p0[p0.conjunto=='teste']; pk=pk[pk.conjunto=='teste']
                        z=p0.merge(pk,on=['indice_dataset','experimento'],suffixes=('_0','_k'),validate='one_to_one')
                        if len(z)!=len(p0) or len(z)!=len(pk) or not np.allclose(z.y_0,z.y_k):
                            raise ValueError('Amostras não pareadas: '+run)
                        for exp,e in z.groupby('experimento'):
                            base=np.mean(abs(e.y_0-e.y_predito_0)); novo=np.mean(abs(e.y_k-e.y_predito_k))
                            rows.append(dict(experimento=exp,config=nomes_curto[run]+f' · K={k}',ganho=100*(base-novo)/base if base>0 else np.nan))
                    if not rows: print('Sem pares completos.'); return
                    r=pd.DataFrame(rows).pivot(index='experimento',columns='config',values='ganho')
                    vals=r.to_numpy(); finite=vals[np.isfinite(vals)]; limite=max(1,abs(finite).max()) if len(finite) else 1
                    fig,ax=plt.subplots(figsize=(max(11,len(r.columns)*1.65),7),layout="constrained"); cmap=plt.get_cmap('RdBu').copy(); cmap.set_bad('#dddddd')
                    im=ax.imshow(np.ma.masked_invalid(vals),cmap=cmap,vmin=-limite,vmax=limite,aspect='auto')
                    ax.set_xticks(range(len(r.columns)),[c.replace(' sem parâmetros','\nsem parâmetros').replace(' + parâmetros','\ncom parâmetros').replace(' · K=','\nK=') for c in r.columns],rotation=0,ha='center',fontsize=10); ax.set_yticks(range(len(r)),r.index)
                    for i in range(len(r)):
                        for j in range(len(r.columns)): ax.text(j,i,f'{vals[i,j]:.1f}%' if np.isfinite(vals[i,j]) else 'N/D',ha='center',va='center',fontsize=10)
                    fig.colorbar(im,ax=ax,label='Redução do MAE de teste (%)\npositivo = melhora',pad=.03,shrink=.85)
                    ax.set_title(f'K selecionado por MAE global de validação | semente {seed.value}')
                    plt.show(); plt.close(fig); display(r); return
                r=d[(d.conjunto=='teste')&(d.colunas==kval.value)]
                fig,ax=plt.subplots(figsize=(14,max(4.5,len(r)*.3)),layout="constrained"); labels=[]
                for (run,etapa),g in r.groupby(['execucao','etapa']):
                    g=g.set_index('semente')
                    if not {42,43}<=set(g.index): continue
                    i=len(labels); labels.append(nomes_curto[run]+f' · {etapa}')
                    ax.plot([g.loc[42,'MAE_mm'],g.loc[43,'MAE_mm']],[i,i],color='gray')
                    ax.scatter(g.loc[42,'MAE_mm'],i,color='#44779c',label='42' if i==0 else None)
                    ax.scatter(g.loc[43,'MAE_mm'],i,color='#d49b56',label='43' if i==0 else None)
                if not labels: plt.close(fig); print('Sem pares completos 42/43 neste K.'); return
                ax.set_yticks(range(len(labels)),labels,fontsize=10); ax.set_xlabel('MAE de teste (mm)'); ax.set_title(f'Variação entre treinamentos | K={kval.value}')
                ax.legend(title='Semente',loc='upper left',bbox_to_anchor=(1.02,1),borderaxespad=0,frameon=False); ax.grid(axis='x',alpha=.2); plt.show(); plt.close(fig)
                print('Mesmas partições; duas sementes fornecem apenas uma comparação descritiva. O filtro de semente não se aplica a este gráfico.')
            except (OSError,KeyError,ValueError) as exc:
                print('Não foi possível montar este gráfico:',exc)
    preencher(); grupo.observe(preencher,names='value')
    for w in [grupo,runs,proto,seed,kval,modo]: w.observe(desenhar,names='value')
    display(widgets.VBox([grupo,runs,widgets.HBox([proto,seed,kval]),modo,out]))
    desenhar()
painel_adicional_tcc()
